# Календарь результатов торговли

Notebook читает экспорт **Closed Trades Report.csv** и строит тёмный календарь:

- зелёный — прибыльный день;
- красный — убыточный день;
- серый — нулевой день или день без сделок;
- внутри дня: чистый P&L, количество сделок и win rate;
- справа: итоги календарных недель;
- сверху: итог месяца и количество торговых дней.

Положите CSV рядом с notebook (или укажите путь в следующей ячейке) и выполните **Run All**.

In [ ]:
from pathlib import Path

# -------------------- НАСТРОЙКИ --------------------
CSV_PATH = Path("Closed Trades Report.csv")

# Для приложенного файла в папке upload; при обычном использовании CSV
# достаточно положить рядом с notebook.
if not CSV_PATH.exists() and Path("upload/Closed Trades Report.csv").exists():
    CSV_PATH = Path("upload/Closed Trades Report.csv")

CLOSE_TIME_COL = "Время закрытия"
ORDER_ID_COL = "Номер ордера"
PNL_COL = "прибыль/убыток"

# Если True, к прибыли/убытку прибавляются платёж, налог, комиссия и своп.
USE_NET_PNL = True
COST_COLS = ["Платеж", "Налог", "Комиссия", "Своп (проценты овернайт)"]

CURRENCY_SYMBOL = "$"
FIRST_WEEKDAY = 0  # 0 = понедельник


In [ ]:
import calendar
import html

import pandas as pd
from IPython.display import HTML, clear_output, display


def _read_report(path: Path) -> pd.DataFrame:
    """Читает отчёт, автоматически пропуская его служебную первую строку."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Файл не найден: {path.resolve()}\n"
            "Положите Closed Trades Report.csv рядом с notebook или измените CSV_PATH."
        )

    with path.open("r", encoding="utf-8-sig") as file:
        first_line = file.readline().strip()
    skiprows = 1 if first_line.startswith("Closed Trades Report") else 0
    return pd.read_csv(path, skiprows=skiprows, encoding="utf-8-sig")


def _to_number(series: pd.Series) -> pd.Series:
    """Преобразует числа из CSV; '--' и пустые значения считает нулём."""
    cleaned = (
        series.astype("string")
        .str.strip()
        .replace({"--": "0", "": "0"})
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(cleaned, errors="coerce").fillna(0.0)


def prepare_trades(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw = _read_report(path)
    required = [CLOSE_TIME_COL, PNL_COL]
    missing = [column for column in required if column not in raw.columns]
    if missing:
        raise KeyError(f"В CSV отсутствуют обязательные колонки: {missing}")

    trades = raw.copy()
    trades["closed_at"] = pd.to_datetime(trades[CLOSE_TIME_COL], errors="coerce")
    invalid_dates = int(trades["closed_at"].isna().sum())
    trades = trades.dropna(subset=["closed_at"]).copy()
    if trades.empty:
        raise ValueError("После разбора дат в отчёте не осталось ни одной сделки.")

    trades["gross_pnl"] = _to_number(trades[PNL_COL])
    trades["costs"] = 0.0
    if USE_NET_PNL:
        for column in COST_COLS:
            if column in trades.columns:
                trades["costs"] += _to_number(trades[column])

    trades["trade_pnl"] = trades["gross_pnl"] + trades["costs"]
    trades["is_win"] = trades["trade_pnl"] > 0
    trades["trade_date"] = trades["closed_at"].dt.date
    trades["month"] = trades["closed_at"].dt.to_period("M").astype(str)

    daily = (
        trades.groupby("trade_date", sort=True)
        .agg(
            pnl=("trade_pnl", "sum"),
            trades=("trade_pnl", "size"),
            wins=("is_win", "sum"),
        )
        .astype({"trades": int, "wins": int})
    )
    daily["win_rate"] = daily["wins"] / daily["trades"] * 100

    trades.attrs["source_path"] = str(path)
    trades.attrs["invalid_dates"] = invalid_dates
    return trades, daily


In [ ]:
RU_MONTHS = [
    "январь", "февраль", "март", "апрель", "май", "июнь",
    "июль", "август", "сентябрь", "октябрь", "ноябрь", "декабрь",
]
RU_WEEKDAYS_SUN = ["Вс", "Пн", "Вт", "Ср", "Чт", "Пт", "Сб"]
RU_WEEKDAYS_MON = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]


def month_label(month: str) -> str:
    period = pd.Period(month, freq="M")
    return f"{RU_MONTHS[period.month - 1].capitalize()} {period.year}"


def format_money(value: float) -> str:
    value = 0.0 if abs(float(value)) < 0.005 else float(value)
    sign = "-" if value < 0 else ""
    absolute = abs(value)
    suffix = ""
    scaled = absolute
    if absolute >= 1_000_000:
        scaled, suffix = absolute / 1_000_000, "M"
    elif absolute >= 1_000:
        scaled, suffix = absolute / 1_000, "K"

    if suffix:
        number = f"{scaled:.2f}".rstrip("0").rstrip(".")
    elif abs(absolute - round(absolute)) < 0.005:
        number = f"{absolute:.0f}"
    else:
        number = f"{absolute:.2f}".rstrip("0").rstrip(".")
    return f"{sign}{CURRENCY_SYMBOL}{number}{suffix}"


def format_percent(value: float) -> str:
    if pd.isna(value):
        return "—"
    if abs(value - round(value)) < 0.05:
        return f"{value:.0f}%"
    return f"{value:.1f}%"


def value_class(value: float) -> str:
    if value > 0.004:
        return "positive"
    if value < -0.004:
        return "negative"
    return "neutral"


def plural_ru(number: int, forms: tuple[str, str, str]) -> str:
    number = abs(int(number)) % 100
    last = number % 10
    if 11 <= number <= 19:
        return forms[2]
    if last == 1:
        return forms[0]
    if 2 <= last <= 4:
        return forms[1]
    return forms[2]


In [ ]:
CALENDAR_CSS = r"""
<style>
.tc-scroll, .tc-scroll * { box-sizing: border-box; }
.tc-scroll {
  width: 100%; max-width: 100%; min-width: 0; overflow: visible; border-radius: 18px;
  font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
}
.tc-static-shell {
  width: 100%; max-width: 100%; min-width: 0; padding: clamp(8px, 1.35vw, 20px);
  border: 1px solid #262d3f; border-radius: 18px;
  background: #101524; color: #f3f5f8;
}
.tc-static-header {
  display: flex; flex-wrap: wrap; align-items: center; justify-content: space-between;
  gap: 10px 18px; margin-bottom: clamp(10px, 1.5vw, 22px);
}
.tc-static-title { font-size: clamp(17px, 1.6vw, 24px); font-weight: 780; letter-spacing: -.025em; }
.tc-static-stats { display: flex; flex-wrap: wrap; align-items: center; gap: 7px; color: #9ba6b8; font-size: clamp(10px, .95vw, 14px); }
.tc-stat-value, .tc-stat-count {
  display: inline-flex; align-items: center; min-height: clamp(28px, 2.5vw, 36px); padding: 5px clamp(7px, .9vw, 13px);
  border-radius: 7px; background: #1b2232; color: #f3f5f8; font-weight: 760;
}
.tc-stat-value.positive { background: #163229; color: #5ee19a; }
.tc-stat-value.negative { background: #3b2028; color: #ff7782; }
.tc-stat-count { background: #3d2d69; color: #dbb8ff; }
.tc-calendar-layout {
  --tc-row-gap: clamp(3px, .6vw, 9px);
  --tc-col-gap: clamp(3px, .65vw, 10px);
  --tc-head-height: clamp(30px, 4vw, 54px);
  --tc-day-height: clamp(66px, 8.3vw, 116px);
  display: grid; width: 100%; max-width: 100%; min-width: 0;
  grid-template-columns: minmax(0, 6.6fr) minmax(0, 1.4fr);
  column-gap: clamp(7px, 1vw, 16px);
}
.tc-days-grid {
  display: grid; min-width: 0; grid-template-columns: repeat(7, minmax(0, 1fr));
  grid-template-rows: var(--tc-head-height) repeat(6, var(--tc-day-height));
  gap: var(--tc-row-gap) var(--tc-col-gap);
}
.tc-weeks-column {
  display: grid; min-width: 0; padding-left: clamp(7px, 1vw, 16px);
  border-left: 1px solid #3a4357;
  grid-template-rows: var(--tc-head-height) repeat(6, var(--tc-day-height));
  row-gap: var(--tc-row-gap);
}
.tc-weekday {
  height: 100%; min-width: 0; display: flex; align-items: center; justify-content: center;
  border: 1px solid #465067; border-radius: 6px; background: #3a4457;
  color: #f4f6f9; font-size: clamp(8px, 1vw, 14px); font-weight: 760;
}
.tc-week-column-spacer { min-width: 0; }
.tc-day, .tc-week-card {
  min-width: 0; min-height: 0; height: 100%; border: 1px solid #333c4f;
  border-radius: clamp(5px, .7vw, 9px);
}
.tc-day {
  position: relative; display: flex; align-items: center; justify-content: center;
  padding: clamp(14px, 1.5vw, 22px) clamp(2px, .55vw, 8px) clamp(5px, .7vw, 10px);
  background: #222a3a; color: #edf1f7; overflow: hidden;
}
.tc-day.weekend { background: #3a4457; border-color: #3a4457; }
.tc-day.outside { background: #181e2d; border-color: #283145; color: #3b4356; }
.tc-day.positive { background: #1d3938; border-color: #55d78c; }
.tc-day.negative { background: #3a202b; border-color: #ff565f; }
.tc-day.neutral-trade { background: #2b303c; border-color: #747b87; }
.tc-day-number {
  color: #eef2f8; font-size: clamp(11px, 1.5vw, 21px); font-weight: 760; line-height: 1;
}
.tc-day-number.corner {
  position: absolute; top: clamp(4px, .65vw, 10px); right: clamp(4px, .7vw, 11px);
  color: #8390a6; font-size: clamp(7px, .75vw, 11px); font-weight: 520;
}
.tc-day.outside .tc-day-number { color: #343c4e; }
.tc-day-content { text-align: center; line-height: 1.25; }
.tc-day-pnl { color: #f2f4f8; font-size: clamp(10px, 1.5vw, 21px); font-weight: 800; letter-spacing: -.015em; white-space: nowrap; }
.tc-day.positive .tc-day-pnl { color: #5ee19a; }
.tc-day.negative .tc-day-pnl { color: #ff7782; }
.tc-day-meta { margin-top: clamp(2px, .35vw, 5px); color: #bdc6d5; font-size: clamp(7px, .9vw, 13px); font-weight: 650; white-space: nowrap; }
.tc-win { margin-top: 2px; font-size: clamp(7px, .75vw, 11px); font-weight: 800; }
.tc-win.good { color: #6cf0ad; }
.tc-win.bad { color: #ff6973; }
.tc-week-card {
  position: relative; display: flex; flex-direction: column; align-items: center; justify-content: center;
  margin-left: 0; padding: clamp(3px, .7vw, 10px);
  background: #30394b; color: #edf1f7; text-align: center; overflow: hidden;
}
.tc-week-card.positive { background: #1d3938; border-color: #55d78c; }
.tc-week-card.negative { background: #3a202b; border-color: #ff565f; }
.tc-week-title { font-size: clamp(7px, .82vw, 12px); font-weight: 760; white-space: nowrap; }
.tc-week-pnl { margin-top: clamp(3px, .45vw, 7px); font-size: clamp(10px, 1.35vw, 19px); font-weight: 800; white-space: nowrap; }
.tc-week-card.positive .tc-week-pnl { color: #5ee19a; }
.tc-week-card.negative .tc-week-pnl { color: #ff7782; }
.tc-week-badge {
  max-width: 100%; margin-top: clamp(3px, .5vw, 8px); padding: clamp(2px, .32vw, 5px) clamp(3px, .52vw, 8px);
  border-radius: 5px; background: #49377a; color: #ddb8ff;
  font-size: clamp(7px, .72vw, 11px); font-weight: 740; white-space: nowrap;
}
.tc-week-card.positive .tc-week-badge,
.tc-week-card.negative .tc-week-badge { background: #49377a; color: #ddb8ff; }
</style>
"""


def month_stats(daily: pd.DataFrame, month: str) -> tuple[float, int, int]:
    period = pd.Period(month, freq="M")
    dates = [date for date in daily.index if date.year == period.year and date.month == period.month]
    if not dates:
        return 0.0, 0, 0
    stats = daily.loc[dates]
    return float(stats["pnl"].sum()), int((stats["trades"] > 0).sum()), int(stats["trades"].sum())


def signed_money(value: float) -> str:
    return ("+" if value > 0.004 else "") + format_money(value)


def six_calendar_weeks(year: int, month_number: int):
    weeks = calendar.Calendar(firstweekday=0).monthdatescalendar(year, month_number)
    while len(weeks) < 6:
        start = pd.Timestamp(weeks[-1][-1]) + pd.Timedelta(days=1)
        weeks.append([(start + pd.Timedelta(days=offset)).date() for offset in range(7)])
    return weeks[:6]


def render_calendar_grid(daily: pd.DataFrame, month: str) -> str:
    period = pd.Period(month, freq="M")
    year, month_number = period.year, period.month
    weeks = six_calendar_weeks(year, month_number)
    day_parts = ['<div class="tc-days-grid">']
    week_parts = ['<div class="tc-weeks-column"><div class="tc-week-column-spacer"></div>']

    for weekday in RU_WEEKDAYS_MON:
        day_parts.append(f'<div class="tc-weekday">{weekday}</div>')

    for week_number, week in enumerate(weeks, start=1):
        in_month_dates = []
        for date in week:
            is_current_month = date.year == year and date.month == month_number
            is_weekend = date.weekday() >= 5
            if is_current_month:
                in_month_dates.append(date)

            if is_current_month and date in daily.index:
                row = daily.loc[date]
                pnl = float(row["pnl"])
                trade_count = int(row["trades"])
                win_rate = float(row["win_rate"])
                win_class = "good" if win_rate > 50 else "bad"
                day_class = "neutral-trade" if value_class(pnl) == "neutral" else value_class(pnl)
                day_parts.append(
                    f'<div class="tc-day {day_class}">'
                    f'<div class="tc-day-number corner">{date.day}</div>'
                    '<div class="tc-day-content">'
                    f'<div class="tc-day-pnl">{html.escape(format_money(pnl))}</div>'
                    f'<div class="tc-day-meta">{trade_count} {plural_ru(trade_count, ("сделка", "сделки", "сделок"))}</div>'
                    f'<div class="tc-win {win_class}">{format_percent(win_rate)}</div>'
                    '</div></div>'
                )
            else:
                classes = ["tc-day"]
                if not is_current_month:
                    classes.append("outside")
                elif is_weekend:
                    classes.append("weekend")
                day_parts.append(
                    f'<div class="{" ".join(classes)}">'
                    f'<div class="tc-day-number">{date.day}</div></div>'
                )

        traded_dates = [date for date in in_month_dates if date in daily.index]
        if traded_dates:
            week_stats = daily.loc[traded_dates]
            weekly_pnl = float(week_stats["pnl"].sum())
            week_days = int((week_stats["trades"] > 0).sum())
            week_trades = int(week_stats["trades"].sum())
        else:
            weekly_pnl, week_days, week_trades = 0.0, 0, 0
        week_parts.append(
            f'<div class="tc-week-card {value_class(weekly_pnl)}">'
            f'<div class="tc-week-title">Неделя {week_number}</div>'
            f'<div class="tc-week-pnl">{html.escape(format_money(weekly_pnl))}</div>'
            f'<div class="tc-week-badge">{week_days} {"day" if week_days == 1 else "days"} | '
            f'{week_trades} {"trade" if week_trades == 1 else "trades"}</div></div>'
        )

    day_parts.append('</div>')
    week_parts.append('</div>')
    return '<div class="tc-calendar-layout">' + "".join(day_parts) + "".join(week_parts) + '</div>'


def static_header(daily: pd.DataFrame, month: str) -> str:
    pnl, trading_days, trades_count = month_stats(daily, month)
    return (
        '<div class="tc-static-header">'
        f'<div class="tc-static-title">{html.escape(month_label(month))}</div>'
        '<div class="tc-static-stats"><span>Итоги месяца:</span>'
        f'<span class="tc-stat-value {value_class(pnl)}">{html.escape(signed_money(pnl))}</span>'
        f'<span class="tc-stat-count">{trading_days} {plural_ru(trading_days, ("день", "дня", "дней"))} | '
        f'{trades_count} {plural_ru(trades_count, ("сделка", "сделки", "сделок"))}</span></div></div>'
    )


def render_month_calendar(daily: pd.DataFrame, month: str) -> str:
    return (
        CALENDAR_CSS + '<div class="tc-scroll"><div class="tc-static-shell">' +
        static_header(daily, month) + render_calendar_grid(daily, month) + '</div></div>'
    )


def show_calendar(daily: pd.DataFrame, month: str) -> None:
    display(HTML(render_month_calendar(daily, month)))


In [ ]:
APP_WIDGET_CSS = r"""
<style>
.tc-app-shell, .tc-app-shell * {
  box-sizing: border-box;
  font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif !important;
}
.tc-app-shell {
  min-width: 0; max-width: 100%; width: 100%; padding: clamp(8px, 1.35vw, 20px); border: 1px solid #262d3f;
  border-radius: 18px; background: #101524; color: #f3f5f8;
  overflow: visible !important;
}
.tc-app-header {
  display: flex !important; flex-wrap: wrap !important; align-items: center !important;
  gap: 7px 10px; width: 100%; min-width: 0; margin-bottom: clamp(10px, 1.5vw, 22px);
}
.tc-app-month { min-width: 0; width: auto !important; }
.tc-app-month-title { padding: 0 clamp(5px, .9vw, 14px); color: #f5f7fa; font-size: clamp(16px, 1.6vw, 23px); font-weight: 800; letter-spacing: -.025em; white-space: nowrap; }
.tc-app-stats { display: flex; flex-wrap: wrap; align-items: center; gap: 7px; color: #9ba6b8; font-size: clamp(10px, .95vw, 14px); white-space: normal; }
.tc-app-stat-value, .tc-app-stat-count {
  display: inline-flex; align-items: center; min-height: clamp(28px, 2.5vw, 36px); padding: 5px clamp(7px, .9vw, 13px);
  border-radius: 7px; background: #1b2232; color: #f3f5f8; font-weight: 760;
}
.tc-app-stat-value.positive { background: #163229; color: #5ee19a; }
.tc-app-stat-value.negative { background: #3b2028; color: #ff7782; }
.tc-app-stat-count { background: #3d2d69; color: #dbb8ff; }
.tc-app-nav button {
  width: clamp(34px, 3.3vw, 48px); height: clamp(34px, 3.2vw, 46px);
  border: 1px solid #365e51 !important; border-radius: 10px !important;
  background: #111a25 !important; color: #65e0a4 !important; font-size: clamp(17px, 1.6vw, 23px) !important;
  font-weight: 450 !important; box-shadow: none !important;
}
.tc-app-nav button:hover { background: #182a2b !important; border-color: #55d78c !important; }
.tc-app-nav button:disabled { opacity: .28; }
.tc-calendar-output, .tc-calendar-output .jupyter-widgets-output-area,
.tc-calendar-output .output, .tc-calendar-output .output_area,
.tc-calendar-output .output_subarea, .tc-calendar-output .jp-OutputArea-output {
  width: 100% !important; max-width: 100% !important; min-width: 0 !important;
  overflow: visible !important;
}
.tc-charts-output, .tc-charts-output .jupyter-widgets-output-area,
.tc-charts-output .output, .tc-charts-output .output_area,
.tc-charts-output .output_subarea, .tc-charts-output .jp-OutputArea-output {
  width: 100% !important; max-width: 100% !important; min-width: 0 !important;
  overflow: visible !important;
}
.tc-charts-grid {
  display: grid; grid-template-columns: repeat(2, minmax(0, 1fr));
  gap: clamp(8px, 1.25vw, 18px); width: 100%; min-width: 0;
  margin-top: clamp(14px, 1.7vw, 24px);
}
.tc-chart-card {
  min-width: 0; padding: clamp(9px, 1.2vw, 17px); border: 1px solid #293247;
  border-radius: 15px; background: #121829; color: #f3f5f8;
}
.tc-chart-title { font-size: clamp(12px, 1.05vw, 15px); font-weight: 780; }
.tc-chart-subtitle { margin-top: 2px; color: #8995a9; font-size: clamp(8px, .72vw, 10px); }
.tc-chart-svg { display: block; width: 100%; height: auto; margin-top: 5px; overflow: visible; }
.tc-chart-axis-text { fill: #a7b1c2; font-size: 10px; font-weight: 600; }
.tc-chart-grid-line { stroke: #394257; stroke-width: 1; stroke-dasharray: 4 5; }
.tc-chart-zero-line { stroke: #748097; stroke-width: 1.2; }
.tc-dashboard {
  width: 100%; min-width: 0; margin-top: clamp(17px, 2vw, 28px);
  padding-top: clamp(4px, .7vw, 10px); color: #f3f5f8;
}
.tc-dashboard-header {
  display: flex; align-items: center; justify-content: space-between; gap: 12px;
  margin-bottom: clamp(10px, 1.35vw, 19px);
}
.tc-dashboard-title { display: flex; align-items: center; gap: 9px; font-size: clamp(20px, 2.2vw, 32px); font-weight: 820; letter-spacing: -.025em; }
.tc-dashboard-sparkle { color: #69dda0; font-size: 1.15em; line-height: 1; }
.tc-dashboard-period { padding: 6px 10px; border-radius: 7px; background: #3d2d69; color: #dbb8ff; font-size: clamp(9px, .8vw, 12px); font-weight: 740; }
.tc-dashboard-grid { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: clamp(7px, 1.15vw, 17px); }
.tc-dashboard-card {
  position: relative; min-width: 0; min-height: clamp(105px, 11vw, 154px);
  padding: clamp(10px, 1.35vw, 20px); border: 1px solid #293247; border-radius: 15px;
  background: #121829; color: #f3f5f8; overflow: visible;
}
.tc-dashboard-card.accent { border-color: #285340; background: #0d1720; }
.tc-metric-label { display: flex; align-items: center; gap: 7px; color: #9dbbe9; font-size: clamp(9px, 1vw, 14px); white-space: nowrap; }
.tc-metric-value { margin-top: clamp(8px, 1vw, 14px); font-size: clamp(18px, 2vw, 29px); font-weight: 820; letter-spacing: -.02em; white-space: nowrap; }
.tc-metric-value.positive { color: #62dea0; }
.tc-metric-value.negative { color: #ef7175; }
.tc-metric-value.neutral { color: #f3f5f8; }
.tc-metric-note { margin-top: 8px; color: #6f80a0; font-size: clamp(8px, .8vw, 11px); }
.tc-card-icon {
  position: absolute; top: clamp(10px, 1.2vw, 18px); right: clamp(10px, 1.2vw, 18px);
  display: flex; align-items: center; justify-content: center; width: clamp(36px, 4vw, 55px);
  height: clamp(36px, 4vw, 55px); border-radius: 13px; background: #12272a;
  color: #62dea0; font-size: clamp(20px, 2.3vw, 32px);
}
.tc-distribution { display: flex; align-items: center; gap: clamp(9px, 1.2vw, 18px); margin-top: 13px; }
.tc-distribution-values { display: flex; gap: 14px; }
.tc-distribution-number { font-size: clamp(18px, 2vw, 29px); font-weight: 820; }
.tc-distribution-caption { margin-top: 2px; color: #9db1d2; font-size: clamp(8px, .8vw, 11px); }
.tc-donut { position: relative; width: clamp(48px, 5.3vw, 72px); aspect-ratio: 1; border-radius: 50%; }
.tc-donut::after { content: ""; position: absolute; inset: 8px; border-radius: 50%; background: #121829; }
.tc-ratio-bar { display: flex; width: 100%; height: 9px; margin-top: 13px; overflow: hidden; border-radius: 999px; background: #30384b; }
.tc-ratio-win { background: #79d99d; }
.tc-ratio-loss { background: #df5e62; }
.tc-ratio-values { display: flex; justify-content: space-between; gap: 8px; margin-top: 7px; font-size: clamp(8px, .8vw, 11px); }
.tc-ratio-win-text { color: #62dea0; }
.tc-ratio-loss-text { color: #ef7175; }
.tc-period-shell {
  margin-bottom: clamp(17px, 2vw, 28px); overflow: visible !important;
}
.tc-period-header {
  display: flex !important; flex-wrap: wrap !important; align-items: center !important;
  justify-content: space-between !important; gap: 10px 18px; width: 100%; min-width: 0;
  margin-bottom: clamp(10px, 1.35vw, 19px); overflow: visible !important;
}
.tc-period-controls {
  display: flex !important; flex-flow: row nowrap !important; align-items: center !important;
  justify-content: flex-end !important; gap: 7px !important;
  width: auto !important; min-width: 269px !important; height: 46px !important;
  min-height: 46px !important; max-height: none !important; overflow: visible !important;
  scrollbar-width: none !important;
}
.tc-period-controls::-webkit-scrollbar { display: none !important; width: 0 !important; height: 0 !important; }
.tc-period-controls > * { flex: 0 0 auto !important; }
.tc-range-button,
.tc-range-button button,
button.tc-range-button,
.tc-range-button.jupyter-button {
  min-width: 54px; height: 36px; padding: 0 12px !important;
  border: 1px solid #39445a !important; border-radius: 7px !important;
  background: #192132 !important; color: #edf2f8 !important;
  font-size: 13px !important; font-weight: 760 !important; line-height: 34px !important;
  box-shadow: none !important; transition: background .16s ease, border-color .16s ease, color .16s ease !important;
}
.tc-range-button:hover,
.tc-range-button button:hover,
button.tc-range-button:hover,
.tc-range-button.jupyter-button:hover {
  border-color: #5fdb9b !important; background: #202b3e !important; color: #75e4ab !important;
}
.tc-range-active,
.tc-range-active button,
button.tc-range-active,
.tc-range-active.jupyter-button {
  border-color: #4d9a79 !important; background: #2b6655 !important; color: #7aebb0 !important;
  box-shadow: inset 0 0 0 1px rgba(122,235,176,.12) !important;
}
.tc-date-range-row {
  display: flex !important; flex-wrap: wrap !important; align-items: center !important;
  gap: 8px 12px !important; width: 100%; margin: 0 0 14px;
  padding: 10px 12px; border: 1px solid #293247; border-radius: 11px; background: #111827;
}
.tc-date-label { color: #9dbbe9; font-size: 12px; font-weight: 700; }
.tc-date-picker { min-width: 155px; }
.tc-date-picker input {
  height: 34px !important; border: 1px solid #3a455b !important; border-radius: 7px !important;
  background: #1a2233 !important; color: #f3f5f8 !important; font: 650 12px Inter, sans-serif !important;
}
.tc-period-output, .tc-period-output .jupyter-widgets-output-area,
.tc-period-output .output, .tc-period-output .output_area,
.tc-period-output .output_subarea, .tc-period-output .jp-OutputArea-output {
  width: 100% !important; max-width: 100% !important; min-width: 0 !important;
  overflow: visible !important;
}
.tc-period-dashboard .tc-dashboard { margin-top: 0; padding-top: 0; }
.tc-period-equity-card {
  width: 100%; min-width: 0; margin-top: clamp(14px, 1.7vw, 24px);
}
.tc-period-equity-card .tc-chart-svg { width: 100%; max-height: 330px; }
@media (max-width: 700px) {
  .tc-period-header { align-items: flex-start !important; }
  .tc-period-controls { justify-content: flex-start !important; }
  .tc-range-button, .tc-range-button button { min-width: 48px; padding: 0 8px !important; }
}
</style>
"""


def app_stats_html(daily: pd.DataFrame, month: str) -> str:
    pnl, trading_days, trades_count = month_stats(daily, month)
    return (
        '<div class="tc-app-stats"><span>Итоги месяца:</span>'
        f'<span class="tc-app-stat-value {value_class(pnl)}">{html.escape(signed_money(pnl))}</span>'
        f'<span class="tc-app-stat-count">{trading_days} {plural_ru(trading_days, ("день", "дня", "дней"))} | '
        f'{trades_count} {plural_ru(trades_count, ("сделка", "сделки", "сделок"))}</span></div>'
    )


def _month_rows(daily: pd.DataFrame, month: str):
    period = pd.Period(month, freq="M")
    dates = sorted(date for date in daily.index if date.year == period.year and date.month == period.month)
    return period, [(date, float(daily.loc[date, "pnl"])) for date in dates]


def _chart_bounds(values):
    values = [float(value) for value in values] + [0.0]
    low, high = min(values), max(values)
    if abs(high - low) < 1e-9:
        padding = max(1.0, abs(high) * 0.15)
        return low - padding, high + padding
    span = high - low
    return (low - span * 0.10 if low < 0 else 0.0), (high + span * 0.10 if high > 0 else 0.0)


def _chart_scaffold(low: float, high: float, days_in_month: int):
    width, height = 560.0, 300.0
    left, right, top, bottom = 62.0, 16.0, 20.0, 38.0
    plot_width, plot_height = width - left - right, height - top - bottom

    def x_for_day(day):
        return left + (float(day) - 1.0) / max(1.0, days_in_month - 1.0) * plot_width

    def y_for_value(value):
        return top + (high - float(value)) / max(1e-9, high - low) * plot_height

    parts = []
    for index in range(5):
        tick = high - (high - low) * index / 4
        y = y_for_value(tick)
        parts.append(f'<line class="tc-chart-grid-line" x1="{left:.1f}" y1="{y:.1f}" x2="{width-right:.1f}" y2="{y:.1f}"/>')
        parts.append(f'<text class="tc-chart-axis-text" x="{left-8:.1f}" y="{y+3.5:.1f}" text-anchor="end">{html.escape(format_money(tick))}</text>')

    tick_days = sorted(set([1, min(8, days_in_month), min(15, days_in_month), min(22, days_in_month), days_in_month]))
    for day in tick_days:
        x = x_for_day(day)
        parts.append(f'<text class="tc-chart-axis-text" x="{x:.1f}" y="{height-12:.1f}" text-anchor="middle">{day}</text>')

    zero_y = y_for_value(0.0)
    parts.append(f'<line class="tc-chart-zero-line" x1="{left:.1f}" y1="{zero_y:.1f}" x2="{width-right:.1f}" y2="{zero_y:.1f}"/>')
    return width, height, plot_width, x_for_day, y_for_value, zero_y, parts


def _equity_curve_svg(rows, days_in_month: int) -> str:
    cumulative = 0.0
    curve = [(1, 0.0)]
    for date, pnl in rows:
        cumulative += pnl
        curve.append((date.day, cumulative))
    low, high = _chart_bounds([value for _, value in curve])
    width, height, _, x_for_day, y_for_value, zero_y, parts = _chart_scaffold(low, high, days_in_month)
    points = [(x_for_day(day), y_for_value(value)) for day, value in curve]
    path = " ".join(("M" if index == 0 else "L") + f" {x:.1f} {y:.1f}" for index, (x, y) in enumerate(points))
    if len(points) > 1:
        area = path + f" L {points[-1][0]:.1f} {zero_y:.1f} L {points[0][0]:.1f} {zero_y:.1f} Z"
        parts.append(f'<path d="{area}" fill="#54c98a" fill-opacity="0.16"/>')
    parts.append(f'<path d="{path}" fill="none" stroke="#60dfa0" stroke-width="2.8" stroke-linecap="round" stroke-linejoin="round"/>')
    for x, y in points[1:]:
        parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="3.2" fill="#60dfa0" stroke="#162b2a" stroke-width="1.2"/>')
    return f'<svg class="tc-chart-svg" viewBox="0 0 {width:.0f} {height:.0f}" role="img" aria-label="Кривая доходности">' + "".join(parts) + '</svg>'


def _daily_bars_svg(rows, days_in_month: int) -> str:
    values = [pnl for _, pnl in rows]
    low, high = _chart_bounds(values)
    width, height, plot_width, x_for_day, y_for_value, zero_y, parts = _chart_scaffold(low, high, days_in_month)
    bar_width = max(4.0, min(17.0, plot_width / max(1, days_in_month) * 0.72))
    for date, pnl in rows:
        x = x_for_day(date.day) - bar_width / 2
        value_y = y_for_value(pnl)
        y = min(value_y, zero_y)
        bar_height = max(1.5, abs(value_y - zero_y))
        color = "#68d99a" if pnl > 0 else "#ee6465" if pnl < 0 else "#7f899b"
        parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_width:.1f}" height="{bar_height:.1f}" rx="2.5" fill="{color}"/>')
    return f'<svg class="tc-chart-svg" viewBox="0 0 {width:.0f} {height:.0f}" role="img" aria-label="Дневной P&amp;L">' + "".join(parts) + '</svg>'


def render_month_charts(daily: pd.DataFrame, month: str) -> str:
    period, rows = _month_rows(daily, month)
    days_in_month = calendar.monthrange(period.year, period.month)[1]
    subtitle = html.escape(month_label(month))
    return (
        '<div class="tc-charts-grid">'
        '<div class="tc-chart-card"><div class="tc-chart-title">Кривая доходности</div>'
        f'<div class="tc-chart-subtitle">{subtitle}</div>{_equity_curve_svg(rows, days_in_month)}</div>'
        '<div class="tc-chart-card"><div class="tc-chart-title">Дневной P&amp;L</div>'
        f'<div class="tc-chart-subtitle">{subtitle}</div>{_daily_bars_svg(rows, days_in_month)}</div>'
        '</div>'
    )


def _metric_label(label: str) -> str:
    return f'<div class="tc-metric-label">{html.escape(label)}</div>'


def _percent(value: float) -> str:
    return f"{value:.1f}%"


def _render_dashboard_for_trades(month_trades: pd.DataFrame, period_label: str, include_header: bool = True) -> str:
    pnl_values = month_trades["trade_pnl"].astype(float) if not month_trades.empty else pd.Series(dtype=float)
    total_trades = int(len(month_trades))
    wins = int((pnl_values > 0).sum())
    losses = int((pnl_values < 0).sum())
    net_pnl = float(pnl_values.sum()) if total_trades else 0.0
    win_rate = wins / total_trades * 100 if total_trades else 0.0
    average_trade = net_pnl / total_trades if total_trades else 0.0

    average_win = float(pnl_values[pnl_values > 0].mean()) if wins else 0.0
    average_loss = abs(float(pnl_values[pnl_values < 0].mean())) if losses else 0.0
    win_loss_ratio = average_win / average_loss if average_loss > 0 else float("inf") if average_win > 0 else 0.0
    ratio_total = average_win + average_loss
    win_share = average_win / ratio_total * 100 if ratio_total > 0 else 50.0
    loss_share = 100.0 - win_share
    distribution_total = wins + losses
    win_angle = wins / distribution_total * 360 if distribution_total else 0.0

    ratio_display = "∞" if win_loss_ratio == float("inf") else f"{win_loss_ratio:.1f}"
    ratio_class = "positive" if win_loss_ratio >= 1 else "negative" if win_loss_ratio > 0 else "neutral"
    donut_background = (
        f"conic-gradient(#79d99d 0deg {win_angle:.1f}deg, #df5e62 {win_angle:.1f}deg 360deg)"
        if distribution_total else "#30384b"
    )

    cards = []
    cards.append(
        '<div class="tc-dashboard-card accent">' +
        _metric_label("Net P&L") +
        f'<div class="tc-metric-value {value_class(net_pnl)}">{html.escape(signed_money(net_pnl))}</div>'
        '<div class="tc-card-icon">$</div></div>'
    )
    cards.append(
        '<div class="tc-dashboard-card accent">' +
        _metric_label("Win Rate %") +
        f'<div class="tc-metric-value {"positive" if win_rate >= 50 else "negative"}">{_percent(win_rate)}</div>'
        '<div class="tc-card-icon">◎</div></div>'
    )
    cards.append(
        '<div class="tc-dashboard-card accent">' +
        _metric_label("Avg Trade") +
        f'<div class="tc-metric-value {value_class(average_trade)}">{html.escape(signed_money(average_trade))}</div>'
        '<div class="tc-card-icon">↗</div></div>'
    )
    cards.append(
        '<div class="tc-dashboard-card">' +
        _metric_label("Total Trades") +
        f'<div class="tc-metric-value neutral">{total_trades}</div></div>'
    )
    cards.append(
        '<div class="tc-dashboard-card">' +
        _metric_label("Win/Loss Distribution") +
        '<div class="tc-distribution"><div class="tc-distribution-values">'
        f'<div><div class="tc-distribution-number" style="color:#62dea0">{wins}</div><div class="tc-distribution-caption">Wins</div></div>'
        f'<div><div class="tc-distribution-number" style="color:#ef7175">{losses}</div><div class="tc-distribution-caption">Losses</div></div>'
        f'</div><div class="tc-donut" style="background:{donut_background}"></div></div></div>'
    )
    cards.append(
        '<div class="tc-dashboard-card">' +
        _metric_label("Avg win/loss ratio") +
        f'<div class="tc-metric-value {ratio_class}">{ratio_display}</div>'
        '<div class="tc-ratio-bar">'
        f'<div class="tc-ratio-win" style="width:{win_share:.1f}%"></div>'
        f'<div class="tc-ratio-loss" style="width:{loss_share:.1f}%"></div></div>'
        '<div class="tc-ratio-values">'
        f'<span class="tc-ratio-win-text">{html.escape(format_money(average_win))}</span>'
        f'<span class="tc-ratio-loss-text">{html.escape(format_money(average_loss))}</span></div></div>'
    )

    header = (
        '<div class="tc-dashboard-header"><div class="tc-dashboard-title">'
        '<span class="tc-dashboard-sparkle">✦</span>Dashboard</div>'
        f'<div class="tc-dashboard-period">{html.escape(period_label)}</div></div>'
    ) if include_header else ""
    return '<div class="tc-dashboard">' + header + '<div class="tc-dashboard-grid">' + "".join(cards) + '</div></div>'


def render_dashboard(trades: pd.DataFrame, daily: pd.DataFrame, month: str) -> str:
    month_trades = trades.loc[trades["month"] == month].copy()
    return _render_dashboard_for_trades(month_trades, month_label(month), include_header=True)


def render_period_dashboard(trades: pd.DataFrame, start_date, end_date) -> str:
    mask = trades["trade_date"].between(start_date, end_date, inclusive="both")
    selected = trades.loc[mask].copy()
    return '<div class="tc-period-dashboard">' + _render_dashboard_for_trades(selected, "", include_header=False) + '</div>'


def _period_equity_curve_svg(trades: pd.DataFrame, start_date, end_date) -> str:
    selected = trades.loc[trades["trade_date"].between(start_date, end_date, inclusive="both")]
    daily_rows = (
        selected.groupby("trade_date", sort=True)["trade_pnl"].sum().items()
        if not selected.empty else []
    )
    cumulative = 0.0
    curve = [(start_date, 0.0)]
    for trade_date, pnl in daily_rows:
        cumulative += float(pnl)
        curve.append((trade_date, cumulative))

    low, high = _chart_bounds([value for _, value in curve])
    width, height = 1120.0, 300.0
    left, right, top, bottom = 70.0, 22.0, 20.0, 40.0
    plot_width, plot_height = width - left - right, height - top - bottom
    total_days = max(1, (end_date - start_date).days)

    def x_for_date(date):
        return left + (date - start_date).days / total_days * plot_width

    def y_for_value(value):
        return top + (high - float(value)) / max(1e-9, high - low) * plot_height

    parts = []
    for index in range(5):
        tick = high - (high - low) * index / 4
        y = y_for_value(tick)
        parts.append(f'<line class="tc-chart-grid-line" x1="{left:.1f}" y1="{y:.1f}" x2="{width-right:.1f}" y2="{y:.1f}"/>')
        parts.append(f'<text class="tc-chart-axis-text" x="{left-8:.1f}" y="{y+3.5:.1f}" text-anchor="end">{html.escape(format_money(tick))}</text>')

    tick_offsets = sorted(set(round(total_days * index / 4) for index in range(5)))
    date_format = "%d.%m.%y" if start_date.year != end_date.year else "%d.%m"
    for offset in tick_offsets:
        tick_date = start_date + pd.Timedelta(days=offset).to_pytimedelta()
        x = x_for_date(tick_date)
        parts.append(f'<text class="tc-chart-axis-text" x="{x:.1f}" y="{height-12:.1f}" text-anchor="middle">{tick_date.strftime(date_format)}</text>')

    zero_y = y_for_value(0.0)
    parts.append(f'<line class="tc-chart-zero-line" x1="{left:.1f}" y1="{zero_y:.1f}" x2="{width-right:.1f}" y2="{zero_y:.1f}"/>')
    points = [(x_for_date(date), y_for_value(value)) for date, value in curve]
    path = " ".join(("M" if index == 0 else "L") + f" {x:.1f} {y:.1f}" for index, (x, y) in enumerate(points))
    if len(points) > 1:
        area = path + f" L {points[-1][0]:.1f} {zero_y:.1f} L {points[0][0]:.1f} {zero_y:.1f} Z"
        parts.append(f'<path d="{area}" fill="#54c98a" fill-opacity="0.16"/>')
    parts.append(f'<path d="{path}" fill="none" stroke="#60dfa0" stroke-width="2.8" stroke-linecap="round" stroke-linejoin="round"/>')
    for x, y in points[1:]:
        parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="3.2" fill="#60dfa0" stroke="#162b2a" stroke-width="1.2"/>')
    return f'<svg class="tc-chart-svg" viewBox="0 0 {width:.0f} {height:.0f}" role="img" aria-label="Кривая доходности выбранного периода">' + "".join(parts) + '</svg>'


def render_period_equity_chart(trades: pd.DataFrame, start_date, end_date) -> str:
    period_text = f"{start_date:%d.%m.%Y} — {end_date:%d.%m.%Y}"
    return (
        '<div class="tc-chart-card tc-period-equity-card">'
        '<div class="tc-chart-title">Кривая доходности</div>'
        f'<div class="tc-chart-subtitle">{html.escape(period_text)}</div>'
        f'{_period_equity_curve_svg(trades, start_date, end_date)}</div>'
    )


def period_dashboard_app(trades: pd.DataFrame):
    if trades.empty:
        raise ValueError("В отчёте нет сделок для Dashboard.")

    first_date = min(trades["trade_date"])
    last_date = max(trades["trade_date"])

    try:
        import ipywidgets as widgets
    except ImportError:
        display(HTML(APP_WIDGET_CSS + '<div class="tc-app-shell tc-period-shell">'
                     '<div class="tc-dashboard-header"><div class="tc-dashboard-title">'
                     '<span class="tc-dashboard-sparkle">✦</span>Dashboard</div></div>'
                     + render_period_dashboard(trades, first_date, last_date)
                     + render_period_equity_chart(trades, first_date, last_date) + '</div>'))
        return None

    state = {"period": "all", "start": first_date, "end": last_date, "syncing_dates": False}
    buttons = {
        "7d": widgets.Button(description="7D", layout=widgets.Layout(width="58px", height="36px")),
        "30d": widgets.Button(description="30D", layout=widgets.Layout(width="62px", height="36px")),
        "90d": widgets.Button(description="90D", layout=widgets.Layout(width="62px", height="36px")),
        "all": widgets.Button(description="All", layout=widgets.Layout(width="58px", height="36px")),
    }
    for button in buttons.values():
        button.add_class("tc-range-button")

    title = widgets.HTML(value='<div class="tc-dashboard-title"><span class="tc-dashboard-sparkle">✦</span>Dashboard</div>')
    controls = widgets.HBox(
        list(buttons.values()),
        layout=widgets.Layout(
            width="269px", min_width="269px", height="46px",
            flex_flow="row nowrap", align_items="center", justify_content="flex-end", overflow="visible",
        ),
    )
    controls.add_class("tc-period-controls")
    header = widgets.HBox([title, controls], layout=widgets.Layout(width="100%", justify_content="space-between", align_items="center"))
    header.add_class("tc-period-header")

    start_label = widgets.HTML(value='<span class="tc-date-label">С:</span>')
    end_label = widgets.HTML(value='<span class="tc-date-label">По:</span>')
    start_picker = widgets.DatePicker(value=first_date, layout=widgets.Layout(width="170px"))
    end_picker = widgets.DatePicker(value=last_date, layout=widgets.Layout(width="170px"))
    start_picker.add_class("tc-date-picker")
    end_picker.add_class("tc-date-picker")
    date_row = widgets.HBox(
        [start_label, start_picker, end_label, end_picker],
        layout=widgets.Layout(width="100%", display="flex", flex_flow="row wrap", align_items="center"),
    )
    date_row.add_class("tc-date-range-row")
    output = widgets.Output(layout=widgets.Layout(width="100%"))
    output.add_class("tc-period-output")

    def set_active(key: str):
        for name, button in buttons.items():
            button.remove_class("tc-range-active")
            if name == key:
                button.add_class("tc-range-active")

    def redraw():
        start_date, end_date = state["start"], state["end"]
        with output:
            clear_output(wait=True)
            if start_date is None or end_date is None:
                display(HTML('<div style="color:#ef7175;padding:12px">Выберите обе даты.</div>'))
            elif start_date > end_date:
                display(HTML('<div style="color:#ef7175;padding:12px">Начальная дата должна быть раньше конечной.</div>'))
            else:
                display(HTML(
                    render_period_dashboard(trades, start_date, end_date)
                    + render_period_equity_chart(trades, start_date, end_date)
                ))

    def choose_period(key: str, days=None):
        start_date = first_date if days is None else max(first_date, last_date - pd.Timedelta(days=days - 1).to_pytimedelta())
        state["period"], state["start"], state["end"] = key, start_date, last_date
        state["syncing_dates"] = True
        start_picker.value = start_date
        end_picker.value = last_date
        state["syncing_dates"] = False
        set_active(key)
        redraw()

    buttons["7d"].on_click(lambda _: choose_period("7d", 7))
    buttons["30d"].on_click(lambda _: choose_period("30d", 30))
    buttons["90d"].on_click(lambda _: choose_period("90d", 90))
    buttons["all"].on_click(lambda _: choose_period("all"))

    def date_changed(_):
        if state["syncing_dates"]:
            return
        state["period"] = "custom"
        state["start"], state["end"] = start_picker.value, end_picker.value
        set_active(None)
        redraw()

    start_picker.observe(date_changed, names="value")
    end_picker.observe(date_changed, names="value")

    shell = widgets.VBox([header, date_row, output], layout=widgets.Layout(width="100%"))
    shell.add_class("tc-app-shell")
    shell.add_class("tc-period-shell")
    set_active("all")
    redraw()
    display(widgets.HTML(value=APP_WIDGET_CSS), shell)
    return state


def calendar_app(trades: pd.DataFrame, daily: pd.DataFrame):
    months = sorted(trades["month"].dropna().unique().tolist())
    if not months:
        raise ValueError("В отчёте нет доступных месяцев.")

    try:
        import ipywidgets as widgets
    except ImportError:
        show_calendar(daily, months[-1])
        display(HTML(APP_WIDGET_CSS + render_month_charts(daily, months[-1])))
        display(HTML(APP_WIDGET_CSS + render_dashboard(trades, daily, months[-1])))
        return None

    state = {"index": len(months) - 1}
    previous_button = widgets.Button(description="‹", layout=widgets.Layout(width="42px", height="42px"))
    next_button = widgets.Button(description="›", layout=widgets.Layout(width="42px", height="42px"))
    previous_button.add_class("tc-app-nav")
    next_button.add_class("tc-app-nav")
    title = widgets.HTML(layout=widgets.Layout(width="auto", min_width="120px"))
    title.add_class("tc-app-month")
    spacer = widgets.Box(layout=widgets.Layout(flex="1 1 auto"))
    stats = widgets.HTML()
    output = widgets.Output(layout=widgets.Layout(width="100%"))
    output.add_class("tc-calendar-output")
    charts_output = widgets.Output(layout=widgets.Layout(width="100%"))
    charts_output.add_class("tc-charts-output")
    dashboard_output = widgets.Output(layout=widgets.Layout(width="100%"))
    dashboard_output.add_class("tc-charts-output")

    def redraw():
        month = months[state["index"]]
        title.value = f'<div class="tc-app-month-title">{html.escape(month_label(month))}</div>'
        stats.value = app_stats_html(daily, month)
        previous_button.disabled = state["index"] == 0
        next_button.disabled = state["index"] == len(months) - 1
        with output:
            clear_output(wait=True)
            display(HTML(CALENDAR_CSS + '<div class="tc-scroll">' + render_calendar_grid(daily, month) + '</div>'))
        with charts_output:
            clear_output(wait=True)
            display(HTML(render_month_charts(daily, month)))
        with dashboard_output:
            clear_output(wait=True)
            display(HTML(render_dashboard(trades, daily, month)))

    def move(step: int):
        state["index"] = max(0, min(len(months) - 1, state["index"] + step))
        redraw()

    previous_button.on_click(lambda _: move(-1))
    next_button.on_click(lambda _: move(1))
    header = widgets.HBox(
        [previous_button, title, next_button, spacer, stats],
        layout=widgets.Layout(width="100%", align_items="center"),
    )
    header.add_class("tc-app-header")
    shell = widgets.VBox([header, output, dashboard_output, charts_output], layout=widgets.Layout(width="100%"))
    shell.add_class("tc-app-shell")
    redraw()
    display(widgets.HTML(value=APP_WIDGET_CSS), shell)
    return state


In [ ]:
trades, daily = prepare_trades(CSV_PATH)
period_dashboard_state = period_dashboard_app(trades)
calendar_selector = calendar_app(trades, daily)
